In [181]:
print("Hello World")

Hello World


Challenge 2 - Gemini RAG with BigQuery
1. Create a Jupyter Notebook using Agent Platform Colab Enterprise.
2. Import the following file into a BigQuery table:
gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv
3. Generate embeddings for each question-and-answer pair, store the embeddings along with
the existing fields.
4. Program a chatbot that can use a vector search to find the data required to answer the
user’s question.
5. Pass the data and question to Gemini to generate an accurate response.
6. Make sure this is coded in Python and well-documented in your Notebook.
7. Upload the Notebook to GitHub for grading.

In [183]:
!gcloud config get-value project

qwiklabs-gcp-02-657b98113afe


In [182]:
!pip install -q google-genai google-cloud-bigquery

In [184]:
from google import genai
from google.genai.types import HttpOptions
from google.cloud import bigquery

PROJECT = "qwiklabs-gcp-02-657b98113afe"
LOCATION = "us-central1"

bq_client = bigquery.Client(project=PROJECT)

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)
## Environment Setup
## Configure Gemini and BigQuery clients.

In [185]:
MODEL = "gemini-2.5-flash"

response = genai_client.models.generate_content(
    model=MODEL,
    contents="Say hello from planet Earth."
)

print(response.text)

## Testing.. having a little fun.

Hello from Planet Earth! It's great to connect.


In [205]:
from google.cloud import bigquery

dataset_id = f"{PROJECT}.aurora_bay"
table_id = f"{PROJECT}.aurora_bay.faqs"

dataset = bigquery.Dataset(dataset_id)
dataset.location = LOCATION
bq_client.create_dataset(dataset, exists_ok=True)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

load_job = bq_client.load_table_from_uri(
    uri,
    table_id,
    job_config=job_config,
)

load_job.result()

table = bq_client.get_table(table_id)

## Load FAQ Data
## Load the Aurora Bay FAQ CSV into BigQuery.


In [206]:
EMBEDDING_MODEL = "text-embedding-005"

def get_embedding(text: str):
    result = genai_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )
    return result.embeddings[0].values

## Generate Embeddings
## Generate embeddings for each FAQ record and store them in BigQuery.

In [188]:
rows = list(
    bq_client.query(
        f"""
        SELECT
            string_field_0 AS question,
            string_field_1 AS answer,
            CONCAT(
                'Question: ',
                string_field_0,
                '\\nAnswer: ',
                string_field_1
            ) AS content
        FROM `{PROJECT}.aurora_bay.faqs`
        """
    ).result()
)

embedded_rows = [
    {
        "question": row.question,
        "answer": row.answer,
        "content": row.content,
        "embedding": get_embedding(row.content)
    }
    for row in rows
]

## Vector Search Retrieval
## Use cosine similarity to identify the most relevant FAQ entries.

In [207]:
print(f"{len(embedded_rows)}")

## Testing

50


In [208]:
import json
from google.cloud import bigquery

embedding_table_id = f"{PROJECT}.aurora_bay.faq_embeddings"

schema = [
    bigquery.SchemaField("question", "STRING"),
    bigquery.SchemaField("answer", "STRING"),
    bigquery.SchemaField("content", "STRING"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
]

bq_client.delete_table(embedding_table_id, not_found_ok=True)
table = bigquery.Table(embedding_table_id, schema=schema)
bq_client.create_table(table)

job_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = bq_client.load_table_from_json(
    embedded_rows,
    embedding_table_id,
    job_config=job_config,
    location=LOCATION,
)

load_job.result()

table = bq_client.get_table(embedding_table_id)

## Generate Embeddings

In [191]:
print(f"{table.num_rows} in {embedding_table_id}")

## Testing

50 in qwiklabs-gcp-02-657b98113afe.aurora_bay.faq_embeddings


In [192]:
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    calc_1 = sum(a * a for a in vec1) ** 0.5
    calc_2 = sum(b * b for b in vec2) ** 0.5

    if calc_1 == 0 or calc_2 == 0:
        return 0

    return dot_product / (calc_1 * calc_2)

    ## Testing the cosine_similarity

In [193]:
def retrieve_faqs(user_question: str, top_k: int = 3):
    question_embedding = get_embedding(user_question)

    query = f"""
    SELECT
        question,
        answer,
        content,
        embedding
    FROM `{PROJECT}.aurora_bay.faq_embeddings`
    """

    rows = list(bq_client.query(query, location=LOCATION).result())

    scored_rows = []

    for row in rows:
        score = cosine_similarity(question_embedding, row.embedding)

        scored_rows.append({
            "question": row.question,
            "answer": row.answer,
            "content": row.content,
            "score": score
        })

    scored_rows.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return scored_rows[:top_k]

    ## Testing the cosine_similarity

In [194]:
matches = retrieve_faqs(
    "Aurora Bay founded?"
)

for match in matches:
    print("Score:", match["score"])
    print("Question:", match["question"])
    print("Answer:", match["answer"])
    print()

    ## Testing the cosine_similarity

Score: 0.8100839042630401
Question: When was Aurora Bay founded?
Answer: Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.

Score: 0.7090300166897706
Question: What is the population of Aurora Bay?
Answer: Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.

Score: 0.6821572746811566
Question: Who is the current mayor of Aurora Bay?
Answer: The current mayor is Linda Greenwood, elected in 2021 for a four-year term.



In [195]:
CHAT_MODEL = "gemini-2.5-flash"

def ask_aurora_bay_chatbot(user_question: str):
    matches = retrieve_faqs(user_question, top_k=3)

    context = "\n\n".join(
        f"Question: {match['question']}\nAnswer: {match['answer']}"
        for match in matches
    )

    prompt = f"""
You are an Aurora Bay FAQ assistant.

Use only the FAQ context below to answer the user's question.
If the answer is not in the context, say you do not have enough information.

FAQ Context:
{context}

User Question:
{user_question}
"""

    response = genai_client.models.generate_content(
        model=CHAT_MODEL,
        contents=prompt
    )

    return response.text

## Gemini Chatbot
## Passed FAQ context and the user question to
## Gemini to generate responses.

In [196]:
print(
    ask_aurora_bay_chatbot(
        "When was Aurora Bay founded?"
    )
)
## Testing

Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.


In [197]:
print(
    ask_aurora_bay_chatbot(
        "Who founded Aurora Bay?"
    )
)
## Testing

Aurora Bay was founded by a group of fur traders.


In [198]:
print(
    ask_aurora_bay_chatbot(
        "What are fur traders?"
    )
)
## Testing

I do not have enough information.


In [199]:
print(
    ask_aurora_bay_chatbot(
        "What is the population of Aurora Bay?"
    )
)
## Testing

Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.


In [200]:
print(
    ask_aurora_bay_chatbot(
        "Does the population of Aurora Bay fluctuate?"
    )
)
## Testing

Yes, the population of Aurora Bay can fluctuate seasonally due to temporary fishing and tourism workforces.


In [201]:
print(
    ask_aurora_bay_chatbot(
        "Does the Aurora Bay get any tourism?"
    )
)
## Testing

Yes, Aurora Bay does get tourism. The population can fluctuate seasonally due to temporary tourism workforces, and visitors arrive via regional flights, ferries, or small cruise ships that make seasonal stops.


In [202]:
print(
    ask_aurora_bay_chatbot(
        "Whats the temperature like in Aurora Bay?"
    )
)
## Testing

Winters in Aurora Bay average between 10°F to 25°F, and summers are milder, around 50°F to 65°F. Temperatures can vary due to coastal weather patterns.


In [203]:
print(
    ask_aurora_bay_chatbot(
        "How far away is Aurora Bay from Idaho?"
    )
)
## Testing

I do not have enough information.


Curiosity Testing

In [204]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
SELECT * FROM `qwiklabs-gcp-02-657b98113afe.aurora_bay.faq_embeddings`
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,question,answer,content,embedding
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...,Question: When was Aurora Bay founded? Answer:...,[-7.95070156e-02 1.16484212e-02 4.04997962e-...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...,Question: What is the population of Aurora Bay...,[-3.23015377e-02 1.56991314e-02 -5.50814308e-...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...,Question: Where is the Aurora Bay Town Hall lo...,[-6.87112436e-02 -2.40753461e-02 -4.58165538e-...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ...",Question: Who is the current mayor of Aurora B...,[-7.59090707e-02 -2.44235918e-02 -2.49287151e-...
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...,Question: What are the primary industries in A...,[-3.48686539e-02 2.40632109e-02 2.75226608e-...
5,Does Aurora Bay have a public library?,Yes. The Aurora Bay Public Library is located ...,Question: Does Aurora Bay have a public librar...,[-0.0521444 -0.02425287 -0.00448025 0.029753...
6,What are the operating hours of the Aurora Bay...,The library is open Monday through Friday from...,Question: What are the operating hours of the ...,[-3.06820478e-02 -5.67120686e-02 -1.50781646e-...
7,How can I apply for a business license in Auro...,Applications for a business license can be sub...,Question: How can I apply for a business licen...,[-4.52062413e-02 -1.80851147e-02 1.35594020e-...
8,Where can I find information about local fishi...,Fishing regulations are published by the Auror...,Question: Where can I find information about l...,[-2.23437231e-02 1.04388017e-02 -1.37174493e-...
9,When does the local fishermen’s market usually...,The fishermen’s market runs every Saturday fro...,Question: When does the local fishermen’s mark...,[-6.45089848e-03 -4.94445488e-03 -3.66270766e-...
